# Tutorial: Multi-Qubit Coupler Workflow

This notebook combines a fast `QCRFGRModel` frequency probe with a compact `FGF1V1Coupling` turn-off-point workflow. The main goal is to show how to scan coupler flux, inspect the coupling surface, and locate the flux bias that turns the effective coupling off.


## Outline

1. Build representative `QCRFGRModel` and `FGF1V1Coupling` examples from the public package exports.
2. Track qubit frequency versus coupler flux and compute one numerical sensitivity point.
3. Sweep effective qubit-qubit coupling versus coupler flux and interpolate the turn-off point.
4. Reuse the structured coupling result to query other target coupling strengths.


In [ ]:
from __future__ import annotations

import importlib.util
from contextlib import redirect_stdout
from io import StringIO

import matplotlib.pyplot as plt
import numpy as np

required = ["numpy", "matplotlib", "qutip"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise ModuleNotFoundError(
        "Install the runtime dependencies first: " + ", ".join(missing)
    )

from pysuqu.qubit import FGF1V1Coupling, QCRFGRModel
from pysuqu.qubit.analysis import (
    analyze_multi_qubit_coupler_sensitivity,
    find_multi_qubit_coupler_detune,
    get_multi_qubit_frequency_at_coupler_flux,
)
from pysuqu.qubit.sweeps import sweep_multi_qubit_coupling_strength_vs_flux_result


## Step 1 - Build representative public models

The `QCRFGRModel` path is a quick way to inspect frequency response. For coupling turn-off analysis we also build a compact `FGF1V1Coupling` example with reduced truncation levels so the public demo stays focused on workflow rather than exhaustive convergence.


In [ ]:
with redirect_stdout(StringIO()):
    qcr_model = QCRFGRModel(
        capacitance_list=[
            70.319e-15,
            90.238e-15,
            6.304e-15 + 9.8e-15,
            78e-15,
            12.65e-15,
        ],
        junc_resis_list=[10007.92, 10007.92 / 6],
        qrcouple=[16.812e-15, 0.0159e-15],
        flux_list=[0.11, 0.11],
        trunc_ener_level=[3, 3],
        is_print=False,
    )

c_j = 9.8e-15
c_q1_total = 165e-15
c_q2_total = 165e-15
c_qc = 23.2e-15
c_q_ground = 5.2e-15
c_qq = 2.1e-15
c_coupler_total = 142e-15

c_11_ground = c_q1_total - c_qc - c_qq - c_q_ground
c_12_ground = c_q2_total - c_q_ground
fgf_capacitance_list = [
    c_11_ground,
    c_12_ground,
    c_q_ground + c_j,
    c_coupler_total - 2 * c_qc + 6 * c_j,
    c_q_ground + c_j,
    c_12_ground,
    c_11_ground,
    c_qq,
    c_qc,
    c_qc,
]

with redirect_stdout(StringIO()):
    fgf_model = FGF1V1Coupling(
        capacitance_list=fgf_capacitance_list,
        junc_resis_list=[7400, 7400 / 6, 7400],
        qrcouple=[18.34e-15, 0.02e-15],
        flux_list=[0.11, 0.11, 0.11],
        trunc_ener_level=[3, 2, 3],
        is_print=False,
    )

{
    "qcr_qubit_f01_ghz": float(qcr_model.qubit_f01),
    "qcr_coupler_f01_ghz": float(qcr_model.coupler_f01),
    "fgf_qubit1_f01_ghz": float(fgf_model.qubit1_f01),
    "fgf_qubit2_f01_ghz": float(fgf_model.qubit2_f01),
    "fgf_baseline_qq_geff_mhz": float(fgf_model.qq_geff * 1e3),
}


## Step 2 - Probe qubit frequency versus coupler flux

This is the fast operating-point view: sweep the coupler flux through a small window, track the qubit frequency, and compute a numerical sensitivity at the nominal bias point.


In [ ]:
coupler_flux_points = np.linspace(0.08, 0.14, 7)
qubit_track = [
    float(get_multi_qubit_frequency_at_coupler_flux(qcr_model, point, qubit_idx=0))
    for point in coupler_flux_points
]

sensitivity_result = analyze_multi_qubit_coupler_sensitivity(
    qcr_model,
    coupler_flux_point=0.11,
    method="numerical",
    flux_step=1e-3,
    qubit_idx=0,
    is_print=False,
    is_plot=False,
)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(coupler_flux_points, qubit_track, marker="o")
ax.set_xlabel("Coupler flux bias")
ax.set_ylabel("Qubit frequency (GHz)")
ax.set_title("QCR frequency response to coupler flux")
ax.grid(alpha=0.3)
plt.show()

{
    "track_min_ghz": float(np.min(qubit_track)),
    "track_max_ghz": float(np.max(qubit_track)),
    "sensitivity_ghz_per_phi0": float(sensitivity_result.sensitivity_value),
    "sensitivity_metadata": dict(sensitivity_result.metadata),
}


## Step 3 - Sweep effective coupling and locate the turn-off point

This is the key coupler workflow. We run a compact `FGF1V1Coupling` scan around the expected off region, collect a structured `CouplingResult`, and then hand that result directly to `find_multi_qubit_coupler_detune(...)` to interpolate the flux bias where the effective coupling crosses zero.

The scan below is intentionally narrow and low-resolution so the tutorial stays runnable; for production calibration work you would widen the search range and then refine around the first zero crossing.


In [ ]:
coupler_flux_axis = np.array([0.20, 0.225, 0.24, 0.245, 0.25])
coupling_result = sweep_multi_qubit_coupling_strength_vs_flux_result(
    fgf_model,
    coupler_flux_axis,
    method="ES",
    is_plot=False,
)
coupling_mhz = np.array(coupling_result.coupling_values) * 1e3
turnoff_flux = float(find_multi_qubit_coupler_detune(coupling_result, coupler_strength=0.0))
turnoff_coupling_mhz = float(np.interp(turnoff_flux, coupling_result.sweep_values, coupling_mhz))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(coupler_flux_axis, coupling_mhz, marker="o", linewidth=2, label="Effective coupling")
ax.axhline(0.0, color="black", linestyle="--", alpha=0.4)
ax.axvline(turnoff_flux, color="tab:red", linestyle=":", alpha=0.8)
ax.scatter([turnoff_flux], [turnoff_coupling_mhz], color="tab:red", zorder=3, label="Turn-off point")
ax.set_xlabel(r"Coupler flux ($\Phi/\Phi_0$)")
ax.set_ylabel("Effective coupling (MHz)")
ax.set_title("FGF coupling strength versus coupler flux")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

{
    "coupler_flux_axis": coupler_flux_axis.tolist(),
    "coupling_mhz": coupling_mhz.tolist(),
    "turnoff_flux_phi0": turnoff_flux,
    "turnoff_coupling_mhz": turnoff_coupling_mhz,
    "baseline_qq_geff_mhz_at_0p11": float(fgf_model.qq_geff * 1e3),
}


## Exercises

Reuse the same `coupling_result` to ask for nonzero target couplings, or widen `coupler_flux_axis` when you want to do a coarse search before zooming into the turn-off region.


In [ ]:
def summarize_target_coupling(target_mhz: float) -> dict[str, float]:
    target_ghz = target_mhz / 1e3
    flux_point = float(find_multi_qubit_coupler_detune(coupling_result, coupler_strength=target_ghz))
    return {
        "target_coupling_mhz": target_mhz,
        "target_flux_phi0": flux_point,
    }

[
    summarize_target_coupling(1.0),
    summarize_target_coupling(0.0),
    summarize_target_coupling(-0.2),
]
